In [1]:
# %% [code]
import os, glob, shutil
import pandas as pd
import ee
import time
import requests
import huggingface_hub
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
huggingface_key = user_secrets.get_secret("huggingface_token")

project = "landslide-identification-nepal" #The googel earth engine project name
input_csv = "/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv" #location of .csv containing landslide incidents
# start_index = 25  #starting range of images to be downloaded
# end_index = 26
upload_batch = 100
repo_id = "sasudo2/landslides"

DOWNLOAD_DIR = '/kaggle/working/downloads'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)   # folder name in your Google Drive
api = huggingface_hub.HfApi(token = huggingface_key)
api.create_repo(repo_id="sasudo2/landslides", repo_type="dataset", exist_ok=True)


gee_key = "/kaggle/input/datasets/sanjayashrestha123/gee-key/landslide-identification-nepal-cccd90850069.json"
service_account = 'kaggle-import@landslide-identification-nepal.iam.gserviceaccount.com'

credentials = ee.ServiceAccountCredentials(service_account, gee_key)

try:
    ee.Initialize(credentials, project=project)
except Exception as e:
    print("EE initialization failed.")
    raise e

csv_filename = input_csv
df = pd.read_csv(csv_filename)
df = df.iloc[721:]
df['incident_on'] = pd.to_datetime(df['incident_on'])

MAX_AOI_DEG = 0.1


def mask_s2_clouds(image):
    scl = image.select('SCL')
    clean_mask = (scl.eq(2).bitwiseOr(scl.eq(4))
                           .bitwiseOr(scl.eq(5))
                           .bitwiseOr(scl.eq(6))
                           .bitwiseOr(scl.eq(7))
                           .bitwiseOr(scl.eq(11)))
    return image.updateMask(clean_mask)

def clamp_aoi(min_lon, min_lat, max_lon, max_lat):
    lon_span = max_lon - min_lon
    lat_span = max_lat - min_lat

    if lon_span <= MAX_AOI_DEG and lat_span <= MAX_AOI_DEG:
        # Area is within limit — return as-is
        return min_lon, min_lat, max_lon, max_lat

    # Area exceeds limit — clamp to MAX_AOI_DEG centered on the bbox center
    cx = (min_lon + max_lon) / 2
    cy = (min_lat + max_lat) / 2
    half = MAX_AOI_DEG / 2
    return cx - half, cy - half, cx + half, cy + half

def download_image(image, aoi, incident_id, filename, scale=10):
    try:
        url = image.getDownloadURL({
            'scale': scale,
            'region': aoi,
            'format': 'GeoTIFF',
            'crs': 'EPSG:4326',
        })
        response = requests.get(url, stream=True, timeout=300)
        response.raise_for_status()
        os.makedirs(f"{DOWNLOAD_DIR}/incident_{incident_id}", exist_ok = True)
        filepath = f'{DOWNLOAD_DIR}/incident_{incident_id}/{filename}.tif'
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded: {filepath}")
    except Exception as e:
        print(f"Failed to download {filename}: {e}")

def submit_landslide_export(incident_id, search_window_days=120):
    row = df[df['id'] == incident_id]
    if row.empty:
        print(f"ID {incident_id} not found.")
        return

    row = row.iloc[0]
    incident_date = row['incident_on']
    c_min_lon, c_min_lat, c_max_lon, c_max_lat = clamp_aoi(
        row['min_lon'], row['min_lat'], row['max_lon'], row['max_lat']
    )
    aoi = ee.Geometry.Rectangle([c_min_lon, c_min_lat, c_max_lon, c_max_lat])

    before_target = incident_date - pd.DateOffset(months=18)
    after_target  = incident_date + pd.DateOffset(months=18)
    half = pd.DateOffset(days=search_window_days // 2)

    before_start = (before_target - half).strftime('%Y-%m-%d')
    before_end   = (before_target + half).strftime('%Y-%m-%d')
    after_start  = (after_target  - half).strftime('%Y-%m-%d')
    after_end    = (after_target  + half).strftime('%Y-%m-%d')

    print(f"\nChecking ID {incident_id}: {row['title']}")

    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
            .map(mask_s2_clouds))

    bands = ['B4', 'B3', 'B2', 'B8']

    periods = [
        ('before', s2.filterDate(before_start, before_end)),
        ('after',  s2.filterDate(after_start,  after_end)),
    ]

    # Validate first
    validated = {}
    for label, collection in periods:
        count = collection.size().getInfo()
        if count == 0:
            print(f"Skipping ID {incident_id} — {label} window has no scenes.")
            return
        print(f"{label}: {count} scene(s) found.")
        validated[label] = collection

    print(f"Downloading images for ID {incident_id}...")

    # Download before/after
    for label, collection in validated.items():
        img = collection.median().select(bands).clip(aoi)
        download_image(img, aoi, incident_id, f'incident_{incident_id}_{label}', scale=10)

    # Download slope
    dem = ee.Image('USGS/SRTMGL1_003')
    slope = ee.Terrain.slope(dem).clip(aoi)
    download_image(slope, aoi, incident_id, f'incident_{incident_id}_slope', scale=30)

def monitor_tasks(tasks, poll_interval=30):
    """Poll submitted tasks until all complete or fail."""
    print(f"\n⏳ Monitoring {len(tasks)} tasks (checking every {poll_interval}s)...")
    pending = list(tasks)

    while pending:
        still_pending = []
        for name, task in pending:
            status = task.status()['state']
            if status == 'COMPLETED':
                print(f"{name}: done")
            elif status == 'FAILED':
                print(f"{name}: FAILED — {task.status().get('error_message', '')}")
            else:
                still_pending.append((name, task))  # READY or RUNNING

        pending = still_pending
        if pending:
            print(f"   {len(pending)} still running...")
            time.sleep(poll_interval)

    print("🎉 All tasks finished.")

# --- Submit all tasks ---
all_tasks = []

# Single incident
# tasks = submit_landslide_export(47070)
# all_tasks.extend(tasks)

# Or batch — first 10

def flush_uploads():
    try:
        api.upload_folder(
            folder_path=DOWNLOAD_DIR,
            repo_id="sasudo2/landslides",
            repo_type="dataset",
            allow_patterns="*.tif",
        )
    except Exception as e:
        print(f"!!!Upload failed, keeping local files: {e}!!!")
        return

    for subdir in glob.glob(f"{DOWNLOAD_DIR}/*/"):
        shutil.rmtree(subdir)
    
    print(f"Uploaded and cleared {DOWNLOAD_DIR}")

from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_WORKERS = 5  # keep modest — GEE enforces per-project concurrent request limits

def process_incident(inc_id):
    submit_landslide_export(inc_id)
    return inc_id

upload_count = 0
incident_ids = df['id'].iloc[:].tolist()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_incident, inc_id): inc_id for inc_id in incident_ids}

    for future in as_completed(futures):
        inc_id = futures[future]
        try:
            future.result()  # raises here if submit_landslide_export threw
        except Exception as e:
            print(f"Incident {inc_id} failed: {e}")

        upload_count += 1
        if upload_count % upload_batch == 0:
            flush_uploads()

if upload_count % upload_batch != 0:
    flush_uploads()

/tmp/ipykernel_16/3782228842.py:39: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['incident_on'] = pd.to_datetime(df['incident_on'])



Checking ID 75193: Landslide at Gahiri Pakha , Phaktanglung Rural Municipality-6
Checking ID 75864: Landslide at Phaktanglung Rural Municipality-6


Checking ID 75849: Landslide at Palata Rural Municipality-7

Checking ID 75302: Landslide at Pathlekhet, Dhulikhel Municipality-11

Checking ID 75618: Landslide at Bahiri Gaam , Sunchhahari Rural Municipality-3
before: 10 scene(s) found.
before: 24 scene(s) found.
before: 10 scene(s) found.
before: 30 scene(s) found.
before: 25 scene(s) found.
after: 5 scene(s) found.
after: 9 scene(s) found.
after: 12 scene(s) found.
after: 24 scene(s) found.
after: 26 scene(s) found.
Downloaded: /kaggle/working/downloads/incident_75302/incident_75302_before.tif
Downloaded: /kaggle/working/downloads/incident_75302/incident_75302_after.tif
Downloaded: /kaggle/working/downloads/incident_75302/incident_75302_slope.tif

Checking ID 75173: Landslide at Bigu Rural Municipality-1
before: 31 scene(s) found.
after: 33 scene(s) found.
Downloaded: /kaggle/working/d

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Downloaded: /kaggle/working/downloads/incident_74731/incident_74731_slope.tif

Checking ID 74736: Landslide at Lele, Godawari_Lalitpur Municipality-5
before: 37 scene(s) found.
after: 34 scene(s) found.
Downloaded: /kaggle/working/downloads/incident_74735/incident_74735_before.tif
Downloaded: /kaggle/working/downloads/incident_74730/incident_74730_after.tif
Downloaded: /kaggle/working/downloads/incident_74730/incident_74730_slope.tif

Checking ID 74737: Landslide at Neupane Gaau , Godawari_Lalitpur Municipality-10
Downloaded: /kaggle/working/downloads/incident_74734/incident_74734_after.tif
before: 37 scene(s) found.
Downloaded: /kaggle/working/downloads/incident_74734/incident_74734_slope.tif

Checking ID 74738: Landslide at Dalchouki , Konjyosom Rural Municipality-4
after: 34 scene(s) found.
before: 37 scene(s) found.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

before: 6 scene(s) found.
Downloaded: /kaggle/working/downloads/incident_38133/incident_38133_before.tif
Uploaded and cleared /kaggle/working/downloads
after: 26 scene(s) found.
Failed to download incident_38048_before: 429 Client Error: Too Many Requests for url: https://earthengine.googleapis.com/v1/projects/landslide-identification-nepal/thumbnails/893155417cb86c3a4463f52b6dfeff2b-cd235c5ae8a4640031228d5d9fdddd91:getPixels
Failed to download incident_38048_after: 429 Client Error: Too Many Requests for url: https://earthengine.googleapis.com/v1/projects/landslide-identification-nepal/thumbnails/6805517543192427e941f5188e5a073f-a5ab8fa7a964d7dc1eb1f0037a3a040e:getPixels
Downloaded: /kaggle/working/downloads/incident_38043/incident_38043_before.tif
Failed to download incident_38048_slope: 429 Client Error: Too Many Requests for url: https://earthengine.googleapis.com/v1/projects/landslide-identification-nepal/thumbnails/d0be780ba1dcd22f878bbdf76b5e0690-de8c534f20dc47a3a0da105b136e7c36

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded and cleared /kaggle/working/downloads
